## tl;dr

方向增强已经按真实5/10/20秒、买卖两侧和配对无信号报价重新测试。保守 through 情景下，MBO 链在07-21和08-07的增量净 PnL 分别为 **+5,033.09** 和 **+1,394.91 HKD**；但独立 MBP 链在07-22仅 **+18.49 HKD**，在08-07为 **−357.17 HKD**。

方向增强没有稳定降低每笔 adverse-selection cost。MBO 的主要改善来源是减少低质量成交和费用周转；MBP 结果不复现。因此目前能确认“方向信号可作为报价/成交过滤器”，不能确认“方向信号稳定改善每笔成交后的价格走势”。

## Context & Methods

### 名词解释

- **Markout**：成交后过5、10或20秒，用未来中价重新评价这笔成交。做市商买入后价格上涨、卖出后价格下跌，markout 为正。
- **Adverse-selection cost（逆向选择成本）**：成交后价格向不利方向移动的幅度。买单成交后价格下跌，或卖单成交后价格上涨，成本为正；数值越低越好。这里剥离了买卖价差，只衡量成交后中价移动。
- **Execution markout**：从真实挂单成交价到未来中价的做市商方向收益，包含挂在 bid/ask 获得的价差。
- **Paired incremental PnL（配对增量 PnL）**：同一天、相同基础距离、库存惩罚、库存上限和撤挂周期下，方向报价 PnL 减去 `signal_strength=0` 的 PnL。
- **Touch / through**：touch 假设成交价触及挂单就成交；through 要求至少穿过一档。前者乐观，后者保守。
- **MBO / MBP**：MBO 从逐订单增删改恢复盘口；MBP 是交易所直接给出的聚合价位数量。两种数据不混合训练或调参。

### Key Assumptions

第二阶段参数选择合并两个更早验证区间，而不是只用07-21：MBO 使用07-09后30%和07-21整日样本外预测；MBP 使用07-21后30%和07-22整日样本外预测。现有数据只有07-21、07-22是相邻交易日，而且只有 MBP 同时覆盖两天，因此仍不等于多周连续验证。

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'output').exists():
    ROOT = ROOT / '0824'
results = json.loads((ROOT / 'output' / 'directional_markout_results.json').read_text(encoding='utf-8'))
validation = json.loads((ROOT / 'output' / 'directional_markout_validation.json').read_text(encoding='utf-8'))
results['as_of'], validation['all_checks_passed']

## Data

MBO 与 MBP 分开形成两条 walk-forward 链。下表保留各日快照数、成交打印数和污染状态。

In [ ]:
quality_rows = []
for family, dates in results['data_quality'].items():
    for date, record in dates.items():
        quality_rows.append({'family': family, 'date': date, **record})
pd.DataFrame(quality_rows)

## Results

### 配对增量 PnL

In [ ]:
pnl_rows = []
for test in results['tests']:
    for fill_mode, comparison in test['paired_incremental_pnl'].items():
        pnl_rows.append({
            'family': test['family'],
            'test_date': test['test_date'],
            'accuracy': test['classification']['accuracy'],
            'fill_mode': fill_mode,
            'validation_dates': ', '.join(test['tuning']['directional']['validation_dates']),
            **comparison,
        })
pnl = pd.DataFrame(pnl_rows)
model_rows = []
for test in results['tests']:
    for fill_mode in ('through', 'touch'):
        model_rows.append({
            'family': test['family'],
            'test_date': test['test_date'],
            'fill_mode': fill_mode,
            'directional_inventory': test['strategies']['directional'][fill_mode]['net_pnl_hkd'],
            'classic_as': test['strategies']['classic_as'][fill_mode]['net_pnl_hkd'],
            'gp_style_discrete': test['strategies']['gp_style_discrete'][fill_mode]['net_pnl_hkd'],
            'paired_no_signal': test['strategies']['paired_no_signal'][fill_mode]['net_pnl_hkd'],
        })
model_pnl = pd.DataFrame(model_rows)
model_pnl.round(2)

### 买卖两侧 adverse-selection cost

`adverse_cost_reduction_bps > 0` 表示方向报价改善；小于0表示变差。

In [ ]:
adverse = pd.DataFrame(results['adverse_selection_deltas'])
through_adverse = adverse[adverse['fill_mode'].eq('through')][[
    'family', 'date', 'side', 'horizon_seconds',
    'directional_adverse_cost_bps', 'paired_no_signal_adverse_cost_bps',
    'adverse_cost_reduction_bps', 'directional_fills', 'paired_no_signal_fills'
]]
through_adverse.round(3)

In [ ]:
markouts = pd.read_csv(ROOT / 'output' / 'directional_markouts.csv')
aggregate = (markouts[markouts['fill_mode'].eq('through')]
    .groupby(['family', 'strategy', 'side', 'horizon_seconds'], as_index=False)
    .agg(fills=('adverse_selection_cost_bps', 'size'),
         mean_adverse_cost_bps=('adverse_selection_cost_bps', 'mean'),
         mean_execution_markout_bps=('execution_markout_bps', 'mean')))
aggregate.round(3)

## Takeaways

1. **MBO 上方向增强有明显增量 PnL**，而且第二次测试使用07-09与07-21两个验证区间后仍在08-07得到正增量。
2. **MBP 不复现**：相邻日07-22只略有改善，08-07为负，说明信号对重建口径或市场状态敏感。
3. **markout 没有稳定全面改善**：不同日期、买卖侧和5/10/20秒的符号会改变。MBO 07-21的主要优势来自方向报价减少成交次数、降低费用和避开部分低质量成交，而不是每一笔成交后的中价都更有利。
4. 当前结论为 `Share with caveats`。要回答是否稳定赚钱，仍需补充至少数周连续、同一 MBO 口径的数据，并冻结参数后按完整周测试。